In [27]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

## Local classes
import importlib
from vocab import Tokenizer
from model_embedding import ModelEmbedding
import rnn

importlib.reload(rnn)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'rnn' from '/Users/krishnan/Nexus/projects/NMT/rnn.py'>

In [28]:
from datasets import load_dataset
import jax
import jax.numpy as jp

In [29]:
ds = load_dataset("shiyue/chr_en", "parallel")

In [44]:
train_set = list(ds["train"]["sentence_pair"])
chr_sentences = [sentence['chr'] for sentence in train_set]
eng_sentences = [sentence['en'] for sentence in train_set]

## Generate vocab for both source and target sentence.
chr_vocab = Tokenizer(chr_sentences, "chr", "chr", "20000")
eng_vocab = Tokenizer(eng_sentences, "eng", "eng", "8000")
random_key = jax.random.key(0)
embed_size = 128
embedding = ModelEmbedding(chr_vocab.get_vocab_size(), eng_vocab.get_vocab_size(), embed_size, random_key)

In [45]:
print(eng_sentences[0])
a, b = eng_vocab.to_numpy([eng_sentences[1]])
print(b)
print(a.shape)
embedding.target_embed(a).shape

and by him every one that believeth is justified from all things, from which ye could not be justified by the law of Moses.
[29]
(1, 29)


(1, 29, 128)

In [57]:
chr_sentences = chr_sentences[:140]
eng_sentences = eng_sentences[:140]

chr_tokens, chr_lengths = chr_vocab.to_numpy(chr_sentences, add_bos=True, add_eos=True)
eng_tokens, eng_lengths = eng_vocab.to_numpy(eng_sentences, add_bos=True, add_eos=True)

rnn.train(chr_tokens, chr_lengths, eng_tokens, eng_lengths, embedding, eng_vocab.get_vocab_size())

INFO:rnn:JAX devices: [MpsDevice(id=0)]
INFO:rnn:JAX backend: mps
INFO:rnn:RNN cumulative params : embedding = 3584000, encoder = 32769, decoder = 1056769 and total = 4673538
INFO:rnn:Loss at epoch 1/10 is 8.987154960632324. Time taken to train = 20.771202167030424
INFO:rnn:Loss at epoch 2/10 is 8.986987113952637. Time taken to train = 34.86444599996321
INFO:rnn:Loss at epoch 3/10 is 8.986677169799805. Time taken to train = 31.64500549994409
INFO:rnn:Loss at epoch 4/10 is 8.98599624633789. Time taken to train = 41.28101712488569
INFO:rnn:Loss at epoch 5/10 is 8.984460830688477. Time taken to train = 50.65555716585368


KeyboardInterrupt: 